# DocsMind lab: dense vs BM25 vs hybrid forum retrieval

Pipeline position: **Query → Embed → Dense search + BM25 → RRF → Thread expansion → Evaluate**.

This lab keeps the chosen GTE embedding model and changes only the search strategy. It evaluates against 16 human-reviewed questions and 35 answer-post labels. Seven unresolved threads remain documented but are not scored.


## Why this experiment matters

A forum question and its answer often use different words. Dense retrieval can match meaning; BM25 can match exact fault codes and part names. Reciprocal Rank Fusion (RRF) combines their rank positions without pretending their incompatible scores are directly comparable.

The final configuration adds a bounded conversation window: each retrieved post brings the next two replies and one previous post. It never looks at answer labels. This tests whether nearby thread context helps without importing an entire 500-page topic.

In an interview, the important answer is not *we used hybrid search*. It is *we measured whether hybrid and context expansion retrieved more human-labelled answers than dense search alone*.


In [ ]:
from collections import defaultdict
from datetime import datetime, timezone
from pathlib import Path
import json
import re
import sys
import time

import faiss
import numpy as np
import pandas as pd
import torch
from IPython.display import display
from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer

ROOT = Path.cwd()
while ROOT != ROOT.parent and not (ROOT / 'docsmind').exists():
    ROOT = ROOT.parent
if not (ROOT / 'docsmind').exists():
    raise RuntimeError('Start Jupyter from the DocsMind repository.')
sys.path.insert(0, str(ROOT))

from docsmind.eval.embedding_retrieval import load_embedding_eval, score_post_rankings
from docsmind.eval.forum_retrieval import collapse_chunks_to_posts, expand_thread_neighbours, rrf_fuse_ids
from docsmind.ingestion.briskoda_chunks import BriskodaChunkConfig, build_briskoda_chunks, load_briskoda_posts

SNAPSHOT_PATH = Path.home() / 'projects/docsmind-data/briskoda/experiments/chunking-v1/snapshot/superb_mk3.briskoda.jsonl'
EVAL_PATH = ROOT / 'data/eval/briskoda_answer_queries.v1.json'
LAB_DIR = Path.home() / 'projects/docsmind-data/briskoda/experiments/hybrid-answer-v1'
RESULTS_PATH = LAB_DIR / 'hybrid-results.json'
LAB_DIR.mkdir(parents=True, exist_ok=True)
assert torch.cuda.is_available(), 'Run this notebook on a DigitalOcean GPU Droplet.'
print('GPU:', torch.cuda.get_device_name(0))


## 1. Build one fixed corpus for both retrieval paths

BM25 indexes every chunk, including lexical-only diagnostic dumps. Dense search embeds only `index_dense=True` chunks. Both paths return post IDs before fusion, preventing a long post from receiving several votes merely because it produced several chunks.


In [ ]:
posts = load_briskoda_posts(SNAPSHOT_PATH)
all_chunks = build_briskoda_chunks(posts, BriskodaChunkConfig())
dense_chunks = [chunk for chunk in all_chunks if chunk['index_dense']]
chunks_by_id = {str(chunk['id']): chunk for chunk in all_chunks}
dense_chunk_ids = [str(chunk['id']) for chunk in dense_chunks]
all_chunk_ids = [str(chunk['id']) for chunk in all_chunks]
posts_by_id = {str(post['post_id']): post for post in posts}
posts_by_topic = defaultdict(list)
for post in posts:
    posts_by_topic[str(post['topic_id'])].append(post)
for topic_posts in posts_by_topic.values():
    topic_posts.sort(key=lambda post: (int(post.get('post_number') or 0), str(post.get('posted_at', ''))))

eval_dataset = load_embedding_eval(EVAL_PATH)
queries = eval_dataset['queries']
questions = [item['question'] for item in queries]
print(f'BM25 chunks: {len(all_chunks):,}')
print(f'Dense chunks: {len(dense_chunks):,}')
print(f'Answerable questions: {len(queries)}')
print(f'Relevant answer posts: {sum(len(item["relevant_post_ids"]) for item in queries)}')


## 2. Build the dense and lexical indexes

GTE runs locally on the GPU and produces normalized 768-dimensional vectors. FAISS Flat uses inner product, which equals cosine similarity after normalization. BM25 tokenizes lower-case words and numbers, preserving exact diagnostic codes such as `P0401`.


In [ ]:
model = SentenceTransformer(
    'Alibaba-NLP/gte-modernbert-base', device='cuda',
    model_kwargs={'torch_dtype': torch.float16},
)
model.max_seq_length = 768
torch.cuda.synchronize()
dense_started = time.perf_counter()
document_vectors = model.encode(
    [chunk['text'] for chunk in dense_chunks], batch_size=32,
    normalize_embeddings=True, convert_to_numpy=True, show_progress_bar=True,
).astype('float32', copy=False)
torch.cuda.synchronize()
dense_build_seconds = time.perf_counter() - dense_started
dense_index = faiss.IndexFlatIP(document_vectors.shape[1])
dense_index.add(document_vectors)

WORD = re.compile(r'[a-z0-9]+')
tokenize = lambda text: WORD.findall(text.lower())
bm25_started = time.perf_counter()
bm25_tokens = [tokenize(chunk['text']) for chunk in all_chunks]
bm25 = BM25Okapi(bm25_tokens)
bm25_build_seconds = time.perf_counter() - bm25_started
print(f'Dense build: {dense_build_seconds / 60:.2f} min')
print(f'BM25 build: {bm25_build_seconds:.2f} sec')


## 3. Retrieve with four configurations

Each retriever produces a post ranking. Dense and BM25 each contribute 100 unique post candidates to RRF. The thread-expanded configuration then adds at most two later replies and one previous post around each hybrid seed. Metrics inspect only the first ten final posts.


In [ ]:
def bm25_chunk_ranking(question, depth=500):
    scores = np.asarray(bm25.get_scores(tokenize(question)))
    depth = min(depth, len(scores))
    positions = np.argpartition(-scores, depth - 1)[:depth]
    positions = positions[np.argsort(-scores[positions], kind='stable')]
    return [all_chunk_ids[position] for position in positions]


def dense_chunk_rankings(question_vectors, depth=300):
    _, positions = dense_index.search(question_vectors, depth)
    return [[dense_chunk_ids[position] for position in row] for row in positions]


torch.cuda.synchronize()
query_started = time.perf_counter()
question_vectors = model.encode(questions, normalize_embeddings=True, convert_to_numpy=True, show_progress_bar=False).astype('float32', copy=False)
torch.cuda.synchronize()
dense_query_seconds = time.perf_counter() - query_started
dense_chunk_results = dense_chunk_rankings(question_vectors)

bm25_query_started = time.perf_counter()
bm25_chunk_results = [bm25_chunk_ranking(question) for question in questions]
bm25_query_seconds = time.perf_counter() - bm25_query_started

rankings = {'gte_dense': [], 'bm25': [], 'hybrid_rrf': [], 'hybrid_thread_window': []}
for dense_ids, lexical_ids in zip(dense_chunk_results, bm25_chunk_results, strict=True):
    dense_posts = collapse_chunks_to_posts(dense_ids, chunks_by_id)[:100]
    lexical_posts = collapse_chunks_to_posts(lexical_ids, chunks_by_id)[:100]
    hybrid_posts = rrf_fuse_ids([dense_posts, lexical_posts], fusion_k=60, top_k=100)
    expanded_posts = expand_thread_neighbours(
        hybrid_posts, posts_by_id, posts_by_topic,
        previous_posts=1, next_posts=2, top_k=100,
    )
    rankings['gte_dense'].append(dense_posts)
    rankings['bm25'].append(lexical_posts)
    rankings['hybrid_rrf'].append(hybrid_posts)
    rankings['hybrid_thread_window'].append(expanded_posts)


## 4. Compare aggregate and per-question evidence

Recall@k measures how much of the human-labelled answer set appears by rank k. MRR@10 rewards placing at least one answer near the top. The per-question table prevents an average from hiding which repair categories still fail.


In [ ]:
summary_rows = []
for name, config_rankings in rankings.items():
    metrics = score_post_rankings(queries, config_rankings, recall_at=(5, 10), mrr_depth=10)
    summary_rows.append({'config': name, **metrics})
summary = pd.DataFrame(summary_rows).sort_values(['recall@10', 'mrr@10'], ascending=False)
display(summary)

diagnostics = []
for query_index, item in enumerate(queries):
    row = {'id': item['id'], 'relevant_posts': len(item['relevant_post_ids'])}
    for name, config_rankings in rankings.items():
        one = score_post_rankings([item], [config_rankings[query_index]], recall_at=(10,), mrr_depth=10)
        row[f'{name}_R@10'] = one['recall@10']
        row[f'{name}_MRR'] = one['mrr@10']
    diagnostics.append(row)
diagnostics_df = pd.DataFrame(diagnostics)
display(diagnostics_df)


In [ ]:
artifact = {
    'created_at': datetime.now(timezone.utc).isoformat(),
    'corpus_version': eval_dataset['corpus_version'],
    'label_status': eval_dataset['label_status'],
    'questions': len(queries),
    'relevant_answer_posts': sum(len(item['relevant_post_ids']) for item in queries),
    'dense_build_seconds': dense_build_seconds,
    'bm25_build_seconds': bm25_build_seconds,
    'dense_query_ms_each': dense_query_seconds * 1000 / len(queries),
    'bm25_query_ms_each': bm25_query_seconds * 1000 / len(queries),
    'summary': summary_rows,
    'per_query': diagnostics,
}
RESULTS_PATH.write_text(json.dumps(artifact, indent=2), encoding='utf-8')
print('Saved:', RESULTS_PATH)
print('Leader:', summary.iloc[0]['config'])


## 5. Decision rule

Adopt hybrid only if it improves answer Recall@10 without an unacceptable MRR drop. Adopt thread expansion only if its gain is broad across queries rather than caused by one lucky adjacent reply. If all configurations remain weak, the next structural experiment is two-stage retrieval: retrieve candidate topics first, then search only inside those topics for answer posts.
